# 03 — Prepare & Export

Build "Without abortion" and "With abortion" comparison tables.

**Steps:**
1. Load mortality data and abortion estimates
2. Create comparison tables (without/with abortion)
3. Export to CSV, Excel, Parquet + codebook


In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
import numpy as np

from src.ingest import load_config
from src.clean_quality import get_connection, run_sql, load_to_duckdb

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

import re
def clean_cause_name(raw_name: str) -> str:
    '''Remove # prefix and ICD codes from cause name.'''
    name = raw_name.strip()
    if name.startswith('#'):
        name = name[1:]
    name = re.sub(r'\s*\([A-Z*][^)]+\)\s*$', '', name)
    return name.strip()


## Step 1: Load Data


In [ ]:
mort_national = run_sql('SELECT * FROM mortality_national ORDER BY deaths DESC', con)
mort_female = run_sql('SELECT * FROM mortality_female ORDER BY deaths DESC', con)
mort_repro = run_sql('SELECT * FROM mortality_female_repro ORDER BY deaths DESC', con)
df_abortions = run_sql('SELECT * FROM abortions', con)

print('Data loaded:')
print(f'  National: {len(mort_national)} causes')
print(f'  Female: {len(mort_female)} causes')
print(f'  Female 15-44: {len(mort_repro)} causes')


## Step 2: Extract Abortion Totals


In [ ]:
abortion_national = df_abortions[df_abortions['measure'] == 'national_total']['value'].iloc[0]
abortion_repro = df_abortions[df_abortions['measure'] == 'repro_age_total']['value'].iloc[0]
abortion_female = abortion_national

print(f'Abortion totals (2024):')
print(f'  National: {abortion_national:,.0f}')
print(f'  Female (all ages): {abortion_female:,.0f}')
print(f'  Female 15-44: {abortion_repro:,.0f}')


## Step 3: Calculate Adjusted Populations


In [ ]:
mort_national['population_adjusted'] = mort_national['population'] + abortion_national
mort_female['population_adjusted'] = mort_female['population'] + abortion_female
mort_repro['population_adjusted'] = mort_repro['population'] + abortion_repro

print('Adjusted populations calculated')


## Step 4: Build "Without abortion" Tables


In [ ]:
def prepare_without(df, category):
    result = df.head(10).copy()
    result['category'] = category
    result['scenario'] = 'Without abortion'
    result['sex'] = 'Both'
    result['gestation_group'] = None
    result['rank'] = range(1, len(result) + 1)
    result['crude_rate_adjusted'] = result['deaths'] / result['population_adjusted'] * 100_000
    
    # Map to shorter display names for charts
    display_map = {
        'Diseases of heart': 'Heart disease',
        'Malignant neoplasms': 'Cancer',
        'Chronic lower respiratory diseases': 'Respiratory disease',
        'Cerebrovascular diseases': 'Stroke',
        'Alzheimer disease': "Alzheimer's",
        'Diabetes mellitus': 'Diabetes',
        'Accidents (unintentional injuries)': 'Accidents',
        'Intentional self-harm (suicide)': 'Suicide',
        'Chronic liver disease and cirrhosis': 'Liver disease',
        'Nephritis, nephrotic syndrome and nephrosis': 'Kidney disease',
        'Influenza and pneumonia': 'Flu/Pneumonia',
        'Essential hypertension and hypertensive renal disease': 'Hypertension',
        'Assault (homicide)': 'Homicide',
        'Pregnancy, childbirth and the puerperium': 'Pregnancy/childbirth',
    }
    result['cause'] = result['cause'].map(display_map).fillna(result['cause'])
    
    return result[['category', 'scenario', 'rank', 'cause_code', 'cause',
                   'deaths', 'sex', 'population', 'population_adjusted',
                   'crude_rate', 'crude_rate_adjusted', 'gestation_group']]

without_national = prepare_without(mort_national, 'National')
without_female = prepare_without(mort_female, 'Female')
without_repro = prepare_without(mort_repro, 'Female 15-44')

print(f'Without tables created:')
print(f'  National: {len(without_national)} rows')
print(f'  Female: {len(without_female)} rows')
print(f'  Female 15-44: {len(without_repro)} rows')


## Step 5: Build "With abortion" Tables

Takes top 10 causes from WITHOUT table, adds abortion row, re-ranks all together.


In [ ]:
def build_with(df_without, abortion_total, category):
    # Start with the WITHOUT table but change scenario to WITH
    df = df_without.copy()
    df['scenario'] = 'With abortion'
    
    # Create abortion row using same population from the first row
    abortion_row = pd.DataFrame([{
        'category': category,
        'scenario': 'With abortion',
        'rank': None,
        'cause_code': 'ABORT',
        'cause': 'Abortion',
        'deaths': int(abortion_total),
        'sex': 'Female',
        'population': df['population'].iloc[0],
        'population_adjusted': df['population_adjusted'].iloc[0],
        'crude_rate': None,
        'crude_rate_adjusted': abortion_total / df['population_adjusted'].iloc[0] * 100_000,
        'gestation_group': '78.6% ≤9w, 14.2% 10-13w, 6.1% 14-20w, 1.1% ≥21w',
    }])
    
    # Combine all rows
    combined = pd.concat([df, abortion_row], ignore_index=True)
    
    # Re-rank by deaths (highest first)
    combined = combined.sort_values('deaths', ascending=False).reset_index(drop=True)
    combined['rank'] = range(1, len(combined) + 1)
    
    return combined

with_national = build_with(without_national, abortion_national, 'National')
with_female = build_with(without_female, abortion_female, 'Female')
with_repro = build_with(without_repro, abortion_repro, 'Female 15-44')

print(f'With tables created:')
print(f'  National: {len(with_national)} rows (abortion rank #{with_national[with_national["cause"] == "Abortion"]["rank"].iloc[0]})')
print(f'  Female: {len(with_female)} rows (abortion rank #{with_female[with_female["cause"] == "Abortion"]["rank"].iloc[0]})')
print(f'  Female 15-44: {len(with_repro)} rows (abortion rank #{with_repro[with_repro["cause"] == "Abortion"]["rank"].iloc[0]})')


## Step 6: Create Master Export Table


In [ ]:
master = pd.concat([without_national, without_female, without_repro,
                    with_national, with_female, with_repro],
                   ignore_index=True)

master['year'] = 2024
master['data_source'] = 'CDC WONDER 2024 + Guttmacher 2024'

# Reorder columns for clarity
master = master[['year', 'category', 'scenario', 'rank', 'cause_code', 'cause',
                 'deaths', 'sex', 'population', 'population_adjusted',
                 'crude_rate', 'crude_rate_adjusted', 'gestation_group', 'data_source']]

print(f'Master table: {len(master)} rows')
print(f'Categories: {master["category"].unique().tolist()}')
print(f'Scenarios: {master["scenario"].unique().tolist()}')
print(f'Sample (first 3 and last 3):')
print(master[['category', 'scenario', 'rank', 'cause', 'deaths']].head(3))
print('...')
print(master[['category', 'scenario', 'rank', 'cause', 'deaths']].tail(3))


## Step 6a: Build Sex & National Chart Tables

Chart tables for Chart 1 (Female vs Male), Chart 3 (National stacked),
and associated detail bars. Uses `cause_display_names` from DuckDB.


In [ ]:
# Load display name mapping
display_map = run_sql('SELECT cause_cdc, cause_display FROM cause_display_names', con)
display_dict = dict(zip(display_map['cause_cdc'], display_map['cause_display']))

def to_display(name):
    """Map cleaned CDC cause name to chart display name."""
    return display_dict.get(name, name)

# --- chart_female_top10 and chart_male_top10 ---
# Get correct total population per sex (sum of distinct age group pops)
sex_pops = {}
for sex in ['Female', 'Male']:
    pop = run_sql(f"""
        SELECT SUM(population) as pop FROM (
            SELECT DISTINCT five_year_age_groups, population
            FROM mortality_sex_age
            WHERE sex = '{sex}'
              AND icd_10_113_cause_list_code = 'GR113-054'
              AND five_year_age_groups != 'Not Stated'
        )
    """, con).iloc[0, 0]
    sex_pops[sex] = pop
    print(f'  {sex} population: {pop:,.0f}')

# Aggregate deaths by sex and cause (leading causes only)
df_sex_mort = run_sql("""
    SELECT sex, icd_10_113_cause_list_code as cause_code,
           icd_10_113_cause_list as cause_raw, SUM(deaths) as deaths
    FROM mortality_sex_age
    WHERE icd_10_113_cause_list LIKE '#%'
      AND five_year_age_groups != 'Not Stated'
    GROUP BY sex, icd_10_113_cause_list_code, icd_10_113_cause_list
""", con)
df_sex_mort['cause'] = df_sex_mort['cause_raw'].apply(clean_cause_name).map(to_display)

# Calculate rates using correct population
df_sex_mort['population'] = df_sex_mort['sex'].map(sex_pops)
df_sex_mort['crude_rate'] = (df_sex_mort['deaths'] / df_sex_mort['population'] * 100_000).round(1)

for sex, tbl_name in [('Female', 'chart_female_top10'), ('Male', 'chart_male_top10')]:
    df_s = df_sex_mort[df_sex_mort['sex'] == sex].sort_values('deaths', ascending=False).head(10)
    chart_df = df_s[['cause', 'crude_rate']].copy()
    chart_df.columns = ['cause', 'rate_per_100k']
    chart_df['total_label'] = chart_df['rate_per_100k'].apply(lambda x: f'{x:.1f}')
    chart_df = chart_df.reset_index(drop=True)
    load_to_duckdb(chart_df, tbl_name, con)
    print(f'  \u2713 {tbl_name}: top={chart_df.iloc[0]["cause"]} ({chart_df.iloc[0]["rate_per_100k"]}/100k)')


In [ ]:
# --- chart_national_stacked: top 5 causes with male/female/abortion split ---
ABORTION_TOTAL = 1_124_000

df_nat = run_sql("""
    SELECT icd_10_113_cause_list as cause_raw,
           SUM(CASE WHEN sex = 'Male' THEN deaths ELSE 0 END) as male_deaths,
           SUM(CASE WHEN sex = 'Female' THEN deaths ELSE 0 END) as female_deaths,
           SUM(deaths) as total_deaths
    FROM mortality_sex_age
    WHERE icd_10_113_cause_list LIKE '#%'
      AND five_year_age_groups != 'Not Stated'
    GROUP BY icd_10_113_cause_list
    ORDER BY total_deaths DESC
    LIMIT 5
""", con)
df_nat['cause'] = df_nat['cause_raw'].apply(clean_cause_name).map(to_display)
df_nat['abortion_deaths'] = 0

# Add abortion row
abort_row = pd.DataFrame([{
    'cause_raw': 'Abortion', 'cause': 'Abortion',
    'male_deaths': 0, 'female_deaths': 0,
    'total_deaths': ABORTION_TOTAL, 'abortion_deaths': ABORTION_TOTAL
}])
df_nat_stacked = pd.concat([df_nat[['cause', 'male_deaths', 'female_deaths', 'total_deaths', 'abortion_deaths']], abort_row[['cause', 'male_deaths', 'female_deaths', 'total_deaths', 'abortion_deaths']]], ignore_index=True)

load_to_duckdb(df_nat_stacked, 'chart_national_stacked', con)
print(f'  ✓ chart_national_stacked: {df_nat_stacked["cause"].tolist()}')


In [ ]:
# --- chart_abortion_gestation: inner segment data for abortion bar ---
gestation = run_sql("""
    SELECT age_group as segment, value as pct
    FROM abortions
    WHERE measure = 'gestation_pct' AND year = 2024
    ORDER BY value DESC
""", con)

# Build x_start, x_end, x_mid for stacked positioning
total_abortions = ABORTION_TOTAL
x_cursor = 0.0
rows = []
for _, r in gestation.iterrows():
    pct = r['pct']
    x_start = x_cursor
    x_end = x_cursor + (pct / 100.0) * total_abortions
    x_mid = (x_start + x_end) / 2
    rows.append({
        'segment': r['segment'].replace('_', ' '),
        'x_start': x_start,
        'x_end': x_end,
        'x_mid': x_mid,
        'pct_label': f'{pct:.0f}%' if pct >= 5 else '',
    })
    x_cursor = x_end

chart_gestation = pd.DataFrame(rows)
load_to_duckdb(chart_gestation, 'chart_abortion_gestation', con)
print(f'  ✓ chart_abortion_gestation: {len(chart_gestation)} segments')
print(chart_gestation[['segment', 'pct_label']].to_string(index=False))


In [ ]:
# --- Detail bar tables (suicide age breakdown, homicide offender race) ---
# chart_male_suicide_age: male suicide by age group
suicide_age = run_sql("""
    SELECT five_year_age_groups as age_group, deaths
    FROM mortality_sex_age
    WHERE sex = 'Male'
      AND icd_10_113_cause_list LIKE '%self-harm (suicide)%'
      AND five_year_age_groups != 'Not Stated'
      AND deaths IS NOT NULL
    ORDER BY deaths DESC
""", con)

# Bin into 4 groups
def age_bin(age_str):
    age_map = {
        '15-19 years': '15-34', '20-24 years': '15-34', '25-29 years': '15-34', '30-34 years': '15-34',
        '35-39 years': '35-54', '40-44 years': '35-54', '45-49 years': '35-54', '50-54 years': '35-54',
        '55-59 years': '55-74', '60-64 years': '55-74', '65-69 years': '55-74', '70-74 years': '55-74',
        '75-79 years': '75+', '80-84 years': '75+', '85+ years': '75+',
    }
    return age_map.get(age_str, 'Other')

suicide_age['bin'] = suicide_age['age_group'].apply(age_bin)
suicide_binned = suicide_age.groupby('bin')['deaths'].sum().reset_index()
suicide_binned = suicide_binned[suicide_binned['bin'] != 'Other'].sort_values('deaths', ascending=False)
total_suicide = suicide_binned['deaths'].sum()

x_cursor = 0.0
detail_rows = []
for _, r in suicide_binned.iterrows():
    pct = r['deaths'] / total_suicide * 100
    x_end = x_cursor + pct
    detail_rows.append({
        'segment': r['bin'], 'x_start': x_cursor, 'x_end': x_end,
        'x_mid': (x_cursor + x_end) / 2, 'pct_label': f'{pct:.0f}%'
    })
    x_cursor = x_end

chart_male_suicide = pd.DataFrame(detail_rows)
load_to_duckdb(chart_male_suicide, 'chart_male_suicide_age', con)

# chart_white_suicide_age: same structure for White panel (same data for now)
load_to_duckdb(chart_male_suicide, 'chart_white_suicide_age', con)
print(f'  ✓ chart_male_suicide_age & chart_white_suicide_age')

# chart_black_homicide_offender: placeholder with known proportions
# Source: FBI SHR 2024 — Black victims, offender race known
homicide_offender = pd.DataFrame([
    {'segment': 'Black offender', 'x_start': 0, 'x_end': 88, 'x_mid': 44, 'pct_label': '88%'},
    {'segment': 'White offender', 'x_start': 88, 'x_end': 97, 'x_mid': 92.5, 'pct_label': '9%'},
    {'segment': 'Other/unknown', 'x_start': 97, 'x_end': 100, 'x_mid': 98.5, 'pct_label': '3%'},
])
load_to_duckdb(homicide_offender, 'chart_black_homicide_offender', con)
print(f'  ✓ chart_black_homicide_offender')
print('✓ All sex/national chart tables built')


## Step 6b: Build Race/Ethnicity Chart Tables

Builds chart_ tables for NH White, NH Black, and Hispanic from the
cleaned aggregated tables. Uses `cause_display_names` for consistent labels.

Guttmacher race proportions (Abortion Patient Survey 2021-2022):
- NH White: 30%
- NH Black: 29%
- Hispanic/Latinx: 30%


In [ ]:
# Guttmacher race proportions
ABORTION_TOTAL = 1_124_000
ABORT_NH_WHITE = int(ABORTION_TOTAL * 0.30)   # 337,200
ABORT_NH_BLACK = int(ABORTION_TOTAL * 0.29)   # 325,960
ABORT_HISPANIC = int(ABORTION_TOTAL * 0.30)   # 337,200

# Load the three aggregated tables
mort_white = run_sql('SELECT * FROM mortality_nh_white ORDER BY deaths DESC', con)
mort_black = run_sql('SELECT * FROM mortality_nh_black ORDER BY deaths DESC', con)
mort_hispanic = run_sql('SELECT * FROM mortality_hispanic ORDER BY deaths DESC', con)

print(f'NH White: {len(mort_white)} causes, pop={mort_white["population"].iloc[0]:,.0f}')
print(f'NH Black: {len(mort_black)} causes, pop={mort_black["population"].iloc[0]:,.0f}')
print(f'Hispanic: {len(mort_hispanic)} causes, pop={mort_hispanic["population"].iloc[0]:,.0f}')


In [ ]:
# --- chart_*_top10: top 10 causes by rate ---
def build_top10(df):
    t = df.head(10)[['cause', 'crude_rate']].copy()
    t.columns = ['cause', 'rate_per_100k']
    t['total_label'] = t['rate_per_100k'].apply(lambda x: f'{x:.1f}')
    return t.reset_index(drop=True)

chart_white_top10 = build_top10(mort_white)
chart_black_top10 = build_top10(mort_black)
chart_hispanic_top10 = build_top10(mort_hispanic)

for tbl, df_tbl in [('chart_white_top10', chart_white_top10),
                     ('chart_black_top10', chart_black_top10),
                     ('chart_hispanic_top10', chart_hispanic_top10)]:
    load_to_duckdb(df_tbl, tbl, con)
    print(f'  ✓ {tbl}: {df_tbl["cause"].tolist()[:5]}...')


In [ ]:
# --- chart_*_percapita: top 5 with abortion inserted, rates adjusted ---
def build_percapita(df, abort_count):
    pop = df['population'].iloc[0]
    adj_pop = pop + abort_count
    rows = []
    for _, r in df.head(5).iterrows():
        rows.append({
            'cause': r['cause'],
            'deaths': int(r['deaths']),
            'rate_per_100k': round(r['deaths'] / adj_pop * 100_000, 1),
            'total_label': str(round(r['deaths'] / adj_pop * 100_000, 1)),
        })
    rows.append({
        'cause': 'Abortion',
        'deaths': abort_count,
        'rate_per_100k': round(abort_count / adj_pop * 100_000, 1),
        'total_label': str(round(abort_count / adj_pop * 100_000, 1)),
    })
    result = pd.DataFrame(rows).sort_values('rate_per_100k', ascending=False).head(5)
    return result.reset_index(drop=True)

chart_white_percapita = build_percapita(mort_white, ABORT_NH_WHITE)
chart_black_percapita = build_percapita(mort_black, ABORT_NH_BLACK)
chart_hispanic_percapita = build_percapita(mort_hispanic, ABORT_HISPANIC)

for tbl, df_tbl in [('chart_white_percapita', chart_white_percapita),
                     ('chart_black_percapita', chart_black_percapita),
                     ('chart_hispanic_percapita', chart_hispanic_percapita)]:
    load_to_duckdb(df_tbl, tbl, con)
    print(f'  ✓ {tbl}')
    print(f'    {df_tbl[["cause", "rate_per_100k"]].to_string(index=False)}')
    print()


In [ ]:
# --- chart_stacked_*: top 5 with male/female/abortion split ---
def build_stacked(df, abort_count):
    rows = []
    for _, r in df.head(5).iterrows():
        rows.append({
            'cause': r['cause'],
            'male_deaths': int(r['male_deaths']),
            'female_deaths': int(r['female_deaths']),
            'total_deaths': int(r['deaths']),
            'abortion_deaths': 0,
        })
    rows.append({
        'cause': 'Abortion',
        'male_deaths': 0,
        'female_deaths': 0,
        'total_deaths': abort_count,
        'abortion_deaths': abort_count,
    })
    return pd.DataFrame(rows)

chart_stacked_white = build_stacked(mort_white, ABORT_NH_WHITE)
chart_stacked_black = build_stacked(mort_black, ABORT_NH_BLACK)
chart_stacked_hispanic = build_stacked(mort_hispanic, ABORT_HISPANIC)

for tbl, df_tbl in [('chart_stacked_white', chart_stacked_white),
                     ('chart_stacked_black', chart_stacked_black),
                     ('chart_stacked_hispanic', chart_stacked_hispanic)]:
    load_to_duckdb(df_tbl, tbl, con)
    print(f'  ✓ {tbl}: {df_tbl["cause"].tolist()}')


In [ ]:
# --- Build chart_annotations table (cross-chart reference data) ---
# Suicide rank for females (from mortality_female which uses raw stripped names)
mort_female = run_sql('SELECT * FROM mortality_female ORDER BY deaths DESC', con)
# mortality_female uses clean_cause_name output (e.g. "Intentional self-harm (suicide)")
suicide_female_row = mort_female[mort_female['cause'].str.contains('suicide', case=False)]
suicide_female_rank = int(suicide_female_row['rank'].iloc[0]) if len(suicide_female_row) > 0 else 99

# Race tables use display names (via cause_display_names table)
homicide_white_rank = int(mort_white[mort_white['cause'] == 'Homicide']['rank'].iloc[0]) if 'Homicide' in mort_white['cause'].values else 99
suicide_black_rank = int(mort_black[mort_black['cause'] == 'Suicide']['rank'].iloc[0]) if 'Suicide' in mort_black['cause'].values else 99

annotations_df = pd.DataFrame([
    {'chart': 'chart_1_sex', 'key': 'suicide_female_rank', 'value': str(suicide_female_rank)},
    {'chart': 'chart_1b_race', 'key': 'homicide_white_rank', 'value': str(homicide_white_rank)},
    {'chart': 'chart_1b_race', 'key': 'suicide_black_rank', 'value': str(suicide_black_rank)},
])
load_to_duckdb(annotations_df, 'chart_annotations', con)
print(f'  Suicide rank for Female: #{suicide_female_rank}')
print(f'  Homicide rank for NH White: #{homicide_white_rank}')
print(f'  Suicide rank for NH Black: #{suicide_black_rank}')
print('\u2713 chart_annotations built')


## Step 7: Export Files


In [ ]:
export_dir = Path('export')
export_dir.mkdir(exist_ok=True)

# CSV
csv_file = export_dir / 'abortion_cause_of_death_v1.csv'
master.to_csv(csv_file, index=False)
print(f'✓ {csv_file.name}')

# Parquet
parquet_file = export_dir / 'abortion_cause_of_death_v1.parquet'
master.to_parquet(parquet_file, index=False)
print(f'✓ {parquet_file.name}')

# Excel
excel_file = export_dir / 'abortion_cause_of_death_v1.xlsx'
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    for cat in ['National', 'Female', 'Female 15-44']:
        cat_data = master[master['category'] == cat]
        sheet_name = cat.replace(' ', '_')
        cat_data.to_excel(writer, sheet_name=sheet_name, index=False)
print(f'✓ {excel_file.name}')

print(f'\nAll files saved to: {export_dir}')


In [ ]:
con.close()
print('✓ Complete')
